# Voice Screening Agent - Colab runner

A voice agent that runs a short senior .NET technical screening end to end:
speak an answer, it transcribes (Egyptian Arabic + English code-mixing handled),
retrieves the relevant rubric criteria, **decides whether to ask one clarifying
follow-up**, then scores 1-5 and speaks the result back in your language.

Everything is open-source and self-hosted. No API keys.

**Before you start:** set the runtime to a GPU.
`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`

Then `Runtime -> Run all`. First run takes ~8-10 minutes, almost all of it
downloading ~10 GB of model weights. The last cell prints a public
`https://....gradio.live` link - open it and the microphone works there.


## 1. Confirm the GPU

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU"
)
print(f"\n{torch.cuda.get_device_name(0)} | "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Clone the repository

In [ ]:
import os, pathlib

REPO_URL = "https://github.com/karimgamalmahmoud/Voice-Agentic-Systems.git"
REPO_DIR = pathlib.Path("/content/Voice-Agentic-Systems")

if REPO_DIR.exists():
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("\nWorking directory:", os.getcwd())

## 3. Install Python dependencies

Torch is deliberately left alone - Colab's build is already CUDA-matched, and
reinstalling it is the fastest way to break the runtime.

In [ ]:
!pip install -q -r requirements.txt

# Colab preinstalls versions that can shadow the ones we need; confirm the
# important three actually imported at the versions we expect.
import transformers, gradio, sentence_transformers
print("transformers", transformers.__version__)
print("gradio", gradio.__version__)
print("sentence-transformers", sentence_transformers.__version__)

## 4. Install and start Ollama

Ollama serves the LLM behind an OpenAI-compatible API. It runs as a separate
process, which keeps the LLM's dependencies completely isolated from the
torch / Whisper / TTS stack above - the single biggest cause of Colab installs
falling over.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, os, requests

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

# Start the server detached; Colab kills foreground background jobs between cells.
log = open("/content/ollama.log", "w")
server = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT)

for attempt in range(60):
    try:
        if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok:
            print(f"Ollama up after {attempt + 1}s")
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start - check /content/ollama.log")

### Pull the model

`qwen2.5:7b-instruct` (~4.7 GB). Chosen for two reasons: it is the strongest
Arabic model that fits a free T4 alongside Whisper and BGE-M3, and it is
reliable at emitting the structured JSON the coverage and scoring stages
depend on.

In [ ]:
!ollama pull qwen2.5:7b-instruct
!ollama list

## 5. Preload the speech and embedding models

Doing this now rather than on the first click means the live demo is not
sitting through a 3 GB download. Expect ~4-6 minutes.

In [ ]:
import sys
sys.path.insert(0, "/content/Voice-Agentic-Systems/src")

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from voice_agent.agent import ScreeningAgent

agent = ScreeningAgent()
agent.warm_up()          # Whisper large-v3 + BGE-M3
ok, msg = agent.llm.health()
print("\nLLM:", msg)
assert ok, msg

## 6. (Optional) Sanity-check transcription first

The riskiest part of this stack is Egyptian Arabic mixed with English technical
terms. Worth confirming before the live demo. Two of the three provided samples
are code-mixed.

In [ ]:
!python scripts/transcribe_samples.py

## 7. Launch the app

Prints a public `gradio.live` URL. Open it in a new tab - the microphone works
through the tunnel, which is what makes this runnable with no local install.

The link stays alive while this cell runs. Stop the cell to shut it down.

In [ ]:
from voice_agent.app import build_ui
import voice_agent.app as app_module

app_module.AGENT = agent      # reuse the models already loaded above
build_ui().launch(share=True)

## 8. (Optional) Run the quality gate

Runs all three provided sample answers through the full pipeline, writes
`docs/QUALITY_GATE_RESULTS.md`, and checks the invariants: retrieval keeps the
two irrelevant reference notes out, Arabic in produces Arabic out, and scores
have not drifted more than a point from the stored baseline.

Stop cell 7 before running this - they compete for GPU memory.

In [ ]:
!python scripts/run_quality_gate.py --no-speak

## 9. (Optional) Unit tests

CPU-only. Covers the follow-up decision policy, corpus chunking, and the
code-mixed language detection - no GPU or model needed.

In [ ]:
!python -m pytest tests/ -q

---

### Troubleshooting

**Gradio link not appearing** - cell 7 must stay running. If it errored, rerun
cell 4's start block; Colab sometimes reaps the Ollama process.

**`model not found`** - rerun the `ollama pull` cell. The pull silently no-ops
if the server was not up yet.

**CUDA out of memory** - `Runtime -> Restart session`, then run all again but
skip cell 6. Whisper large-v3, BGE-M3 and Qwen-7B together sit around 10 GB of
the T4's 15 GB, so there is headroom, but a stale session can hold weights.

**Microphone blocked** - the browser needs permission on the `gradio.live`
origin, not on the Colab tab. Look for the mic icon in the address bar.
